# 🌱 Karat Farms — Asset Tagger
Uses Qwen2-VL to analyse images/videos from Cloudinary and generate tags + descriptions.

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

**Models available:**
- `Qwen/Qwen2-VL-2B-Instruct` — fast, fits easily in T4 VRAM (default, Cell 3A)
- `Qwen/Qwen2-VL-7B-Instruct` — better quality; set `USE_7B_MODEL = True` in Cell 3B and run it manually

In [ ]:
# CELL 1 — Install dependencies
!pip install transformers accelerate qwen-vl-utils torch opencv-python-headless requests bitsandbytes -q
print('✅ Dependencies installed')

In [ ]:
# CELL 2 — Imports
import os
# Must be set before CUDA initialises (first CUDA call is model load in Cell 3A)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import requests
import cv2
import numpy as np
import json
import re
import gc
import logging
import time
import traceback
from PIL import Image
from io import BytesIO
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

print('✅ Imports done')
print(f'PYTORCH_CUDA_ALLOC_CONF = {os.environ["PYTORCH_CUDA_ALLOC_CONF"]}')

In [ ]:
# CELL 2B — Logging & timing utilities
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    force=True
)
log = logging.getLogger('kf_tagger')


def log_gpu():
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserv = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        log.info(f'GPU  allocated={alloc:.2f}GB  reserved={reserv:.2f}GB  total={total:.2f}GB')
    else:
        log.warning('No CUDA GPU detected — running on CPU')


class Timer:
    def __init__(self, label):
        self.label = label
        self.elapsed = 0.0

    def __enter__(self):
        self._start = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.time() - self._start
        log.info(f'[TIMER] {self.label}: {self.elapsed:.2f}s')


print('✅ Logging ready')

In [ ]:
# CELL 3A — Load 2B model (DEFAULT — run this, skip Cell 3B)
MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'
log.info(f'Loading model: {MODEL_ID}')

with Timer('Model + processor load'):
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto'
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)

log_gpu()
print('✅ Model loaded (2B)')

In [ ]:
# CELL 3B — Load 7B model in 4-bit (OPTIONAL — run manually, do NOT use Run All)
# Set USE_7B_MODEL = True then run this cell on its own
USE_7B_MODEL = False

if not USE_7B_MODEL:
    print('⏭️  Cell 3B skipped — 2B model active (set USE_7B_MODEL = True to switch)')
else:
    if 'model' in dir():
        del model
        torch.cuda.empty_cache()
        gc.collect()
        log.info('Freed previous model from GPU')

    MODEL_ID = 'Qwen/Qwen2-VL-7B-Instruct'
    log.info(f'Loading model: {MODEL_ID} (4-bit quantised)')

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    with Timer('Model + processor load (7B 4-bit)'):
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map='auto'
        )
        processor = AutoProcessor.from_pretrained(MODEL_ID)

    log_gpu()
    print('✅ Model loaded (7B 4-bit)')

In [ ]:
# CELL 4 — Hardcoded Karat Farms test assets
ASSETS = [
    {
        'id': 'u6uosvjhybwofoqyzczo',
        'type': 'video',
        'url': 'https://res.cloudinary.com/dnfmufn2l/video/upload/v1782039077/karat_farms/broll/u6uosvjhybwofoqyzczo.mp4'
    },
    {
        'id': 'cbhk3f1wmqdhvpbybrit',
        'type': 'image',
        'url': 'https://res.cloudinary.com/dnfmufn2l/image/upload/v1782038996/karat_farms/broll/cbhk3f1wmqdhvpbybrit.jpg'
    },
    {
        'id': 'gfdymkknli0ykjskupsy',
        'type': 'video',
        'url': 'https://res.cloudinary.com/dnfmufn2l/video/upload/v1782038910/karat_farms/broll/gfdymkknli0ykjskupsy.mp4'
    },
]

log.info(f'{len(ASSETS)} assets queued: {[a["id"] for a in ASSETS]}')
print(f'✅ {len(ASSETS)} assets queued')

In [ ]:
# CELL 5 — Helper functions

# Max pixels per image/frame fed to the model.
# Qwen2-VL uses 28x28 pixel patches; tokens = MAX_PIXELS / (28*28).
# 512 * 28 * 28 = 401,408 px → 512 tokens per frame (2 frames = 1024 visual tokens total)
MAX_PIXELS = 512 * 28 * 28


def resize_for_model(img):
    w, h = img.size
    if w * h > MAX_PIXELS:
        scale = (MAX_PIXELS / (w * h)) ** 0.5
        img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.LANCZOS)
        log.info(f'  Resized {w}x{h} → {img.size[0]}x{img.size[1]} ({img.size[0]*img.size[1]:,}px)')
    return img


def download_file(url):
    log.info(f'Downloading: {url}')
    t0 = time.time()
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    data = response.content
    elapsed = time.time() - t0
    log.info(f'  → {len(data) / 1e6:.2f} MB in {elapsed:.2f}s')
    return data


def extract_video_frames(video_bytes, num_frames=2):
    temp_path = '/tmp/temp_video.mp4'
    with open(temp_path, 'wb') as f:
        f.write(video_bytes)

    cap = cv2.VideoCapture(temp_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = total_frames / fps if fps > 0 else 0
    log.info(f'  Video: {total_frames} frames  {fps:.1f} fps  {duration:.1f}s duration')

    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        else:
            log.warning(f'  Failed to read frame at index {idx}')

    cap.release()
    log.info(f'  → Extracted {len(frames)}/{num_frames} frames')
    return frames


def analyse_asset(asset):
    log.info(f'── START {asset["id"]} ({asset["type"]}) ──')
    asset_start = time.time()

    try:
        if asset['type'] == 'image':
            raw = download_file(asset['url'])
            images = [resize_for_model(Image.open(BytesIO(raw)).convert('RGB'))]
            log.info(f'  Image size after resize: {images[0].size}')
        else:
            raw = download_file(asset['url'])
            with Timer('Frame extraction'):
                frames = extract_video_frames(raw, num_frames=2)
            images = [resize_for_model(f) for f in frames]

        content = [{'type': 'image'} for _ in images]
        content.append({
            'type': 'text',
            'text': (
                'Describe this image in 2-3 factual sentences. '
                'Only describe what you directly observe — no explanations, no reasons, no guesses. '
                'Cover: what plants or crops you see, the setting (rooftop/indoor/outdoor), '
                'containers or structures, any people and what they are doing, and the lighting.'
            )
        })

        messages = [{'role': 'user', 'content': content}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        with Timer('Tokenise + move to device'):
            inputs = processor(text=[text], images=images, return_tensors='pt').to(model.device)

        n_tokens = inputs['input_ids'].shape[1]
        log.info(f'  Input tokens: {n_tokens}')
        log_gpu()

        with Timer('Model inference'):
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    repetition_penalty=1.5,
                )

        input_len = inputs['input_ids'].shape[1]
        raw_out = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

        # Free GPU tensors immediately
        del inputs, output
        torch.cuda.empty_cache()

        preview = raw_out[:200] + ('...' if len(raw_out) > 200 else '')
        log.info(f'  Raw output: {preview}')

        total_time = time.time() - asset_start
        log.info(f'── END {asset["id"]} in {total_time:.2f}s ──')
        return raw_out

    except Exception as e:
        log.error(f'Error analysing {asset["id"]}: {e}')
        log.error(traceback.format_exc())
        raise


print(f'✅ Helper functions ready  (MAX_PIXELS={MAX_PIXELS:,}  →  ~{MAX_PIXELS//(28*28)} tokens/frame)')

In [ ]:
# CELL 6 — Run on all assets and print results

results = []
run_start = time.time()

for i, asset in enumerate(ASSETS, 1):
    log.info(f'\n{"=" * 60}')
    log.info(f'Asset {i}/{len(ASSETS)}: {asset["id"]}')
    log.info('=' * 60)

    torch.cuda.empty_cache()
    gc.collect()

    try:
        description = analyse_asset(asset)
        log.info(f'  Description ({len(description.split())} words): {description[:120]}...')

        result = {
            'id': asset['id'],
            'type': asset['type'],
            'url': asset['url'],
            'description': description,
        }
        results.append(result)

        print(f'\n📋 {asset["id"]} ({asset["type"]}):')
        print(f'   {description}')
        print('─' * 60)

    except Exception as e:
        log.error(f'Asset {asset["id"]} failed — skipping. Error: {e}')
        results.append({'id': asset['id'], 'type': asset['type'], 'error': str(e)})

total_elapsed = time.time() - run_start
avg = total_elapsed / len(ASSETS) if ASSETS else 0
log.info(f'Run complete: {total_elapsed:.2f}s total  {avg:.2f}s avg per asset')
print('\n✅ All done!')
print('\n📦 Full results:')
print(json.dumps(results, indent=2))